# Calgary Traffic Risk Intelligence

## 02 — Exploratory Data Analysis

This notebook explores the cleaned Calgary traffic incident data before
performing geographic clustering or predictive modelling.

### Objectives

- Load and validate the cleaned datasets
- Understand the overall incident history
- Analyze yearly, monthly, weekly, and hourly patterns
- Compare rush-hour and seasonal patterns
- Inspect common incident descriptions
- Explore geographic distribution
- Verify coordinates before hotspot clustering

> Important: Incident counts represent recorded incident frequency.
> They should not be interpreted directly as crash probability or road
> safety risk because traffic-volume/exposure data is not included.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Display useful DataFrame information clearly.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [2]:
# Find the project root.
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"

INCIDENTS_FILE = CLEAN_DIR / "incidents_clean.csv"
SIGNALS_FILE = CLEAN_DIR / "signals_clean.csv"

print("Project root:", PROJECT_ROOT)
print("Incidents file exists:", INCIDENTS_FILE.exists())
print("Signals file exists:", SIGNALS_FILE.exists())

missing_inputs = [
    path for path in [INCIDENTS_FILE, SIGNALS_FILE]
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Run 01_data_preparation.ipynb first. "
        f"Missing: {missing_inputs}"
    )

Project root: c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence
Incidents file exists: True
Signals file exists: True


## Load Cleaned Data

This notebook uses the cleaned datasets produced by
`01_data_preparation.ipynb`.

The raw files are not loaded again because cleaning should happen in
only one place.

In [3]:
# Load cleaned data.
incidents = pd.read_csv(INCIDENTS_FILE)
signals = pd.read_csv(SIGNALS_FILE)

print(
    f"Incidents: {incidents.shape[0]:,} rows × "
    f"{incidents.shape[1]} columns"
)

print(
    f"Signals: {signals.shape[0]:,} rows × "
    f"{signals.shape[1]} columns"
)

Incidents: 64,798 rows × 20 columns
Signals: 1,791 rows × 13 columns


In [4]:
# Restore datetime columns after reading from CSV.
incidents["start_dt_utc"] = pd.to_datetime(
    incidents["start_dt_utc"],
    utc=True
)

incidents["start_dt_local"] = pd.to_datetime(
    incidents["start_dt_local"],
    utc=True
).dt.tz_convert("America/Edmonton")

incidents["modified_dt_utc"] = pd.to_datetime(
    incidents["modified_dt_utc"],
    utc=True,
    errors="coerce"
)

signals["install_date"] = pd.to_datetime(
    signals["install_date"],
    errors="coerce"
)

print("Datetime conversion complete.")

Datetime conversion complete.


In [5]:
# Preview both datasets.
display(incidents.head())
display(signals.head())

,incident_id,incident_location,description,quadrant,start_dt_utc,start_dt_local,modified_dt_utc,date_local,year,month,month_name,day,weekday,weekday_number,hour,weekend,season,time_period,latitude,longitude
0,2016-12-06 17:00:00+00:00-114.064987451.09611149,Eastbound McKnight Boulevard at 2 Street NW,Multi vehicle incident.,NW,2016-12-06 17:00:00+00:00,2016-12-06 10:00:00-07:00,2016-12-06 17:01:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,10,False,Winter,Midday,51.096111,-114.064987
1,2016-12-06 21:36:00+00:00-114.020547951.0476343,Eastbound Memorial Drive approaching Deerfoot ...,2 vehicle incident blocking the middle lane.,NE,2016-12-06 21:36:00+00:00,2016-12-06 14:36:00-07:00,2016-12-06 21:42:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,14,False,Winter,Midday,51.047634,-114.020548
2,2016-12-06 23:25:00+00:00-114.058178551.04447099,Macleod Trail at 9 Avenue SE,2 vehicle incident.,SE,2016-12-06 23:25:00+00:00,2016-12-06 16:25:00-07:00,2016-12-06 23:26:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,16,False,Winter,Evening Rush,51.044471,-114.058178
3,2016-12-06 23:26:00+00:00-114.030871751.02839263,Ogden Road at Bonnybrook Road SE,2 vehicle incident.,SE,2016-12-06 23:26:00+00:00,2016-12-06 16:26:00-07:00,2016-12-06 23:38:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,16,False,Winter,Evening Rush,51.028393,-114.030872
4,2016-12-07 00:05:00+00:00-114.119583551.06639113,Southbound University Drive at Crowchild Trail NW,2 vehicle incident.,NW,2016-12-07 00:05:00+00:00,2016-12-06 17:05:00-07:00,2016-12-07 00:10:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,17,False,Winter,Evening Rush,51.066391,-114.119584


,signal_id,mist_no,install_date,first_road,second_road,intersection_name,quadrant,control_type,ped_buttons,ped_timer,accessible_pedestrian_signal,latitude,longitude
0,1109000525,0909,2000-11-02,CROWCHILD TRAIL,12 MILE COULEE ROAD,CROWCHILD TRAIL & 12 MILE COULEE ROAD,NW,Traffic signal,Yes,Yes,Yes,51.143437,-114.256977
1,1109000526,0892,2006-01-11,TUSCANY BOULEVARD,TUSCANY WAY,TUSCANY BOULEVARD & TUSCANY WAY,NW,Traffic signal,Yes,No,No,51.123965,-114.246059
2,1109000527,0891,2005-08-16,TUSCANY BOULEVARD,TUSCANY VALLEY VIEW,TUSCANY BOULEVARD & TUSCANY VALLEY VIEW,NW,Traffic signal,Yes,No,No,51.124997,-114.242822
3,1109000528,0868,2007-05-07,5029,NOSE HILL DRIVE,5029 & NOSE HILL DRIVE,NW,Traffic signal T intersection,Yes,No,No,51.107086,-114.239813
4,1109000529,0890,2005-10-06,TUSCANY BOULEVARD,TUSCANY SPRINGS BOULEVARD,TUSCANY BOULEVARD & TUSCANY SPRINGS BOULEVARD,NW,Traffic signal T intersection,Yes,No,No,51.125473,-114.238610


## Initial Validation

Before analysis, verify that the cleaned data still satisfies the main
assumptions established during data preparation.

In [6]:
# Check key IDs, timestamps, and coordinates.
assert incidents["incident_id"].is_unique
assert signals["signal_id"].is_unique

assert incidents["start_dt_local"].notna().all()

assert incidents["latitude"].notna().all()
assert incidents["longitude"].notna().all()

assert signals["latitude"].notna().all()
assert signals["longitude"].notna().all()

print("Validation passed.")

Validation passed.


## Dataset Overview

Start with high-level information about the time period, number of
incidents, traffic controls, and available incident descriptions.

In [7]:
# Build a simple dataset summary.
start_date = incidents["start_dt_local"].min()
end_date = incidents["start_dt_local"].max()

summary = pd.DataFrame(
    {
        "Metric": [
            "Total incidents",
            "Traffic controls",
            "Unique incident descriptions",
            "Earliest incident",
            "Latest incident",
        ],
        "Value": [
            f"{len(incidents):,}",
            f"{len(signals):,}",
            f"{incidents['description'].nunique():,}",
            str(start_date),
            str(end_date),
        ],
    }
)

display(summary)

,Metric,Value
0,Total incidents,"64,798"
1,Traffic controls,"1,791"
2,Unique incident descriptions,"6,654"
3,Earliest incident,2016-12-06 10:00:00-07:00
4,Latest incident,2026-09-28 06:35:14-06:00


In [8]:
# Count incidents by year.
yearly_counts = (
    incidents
    .groupby("year")
    .size()
    .reset_index(name="incident_count")
)

display(yearly_counts)

,year,incident_count
0,2016,311
1,2017,5273
2,2018,6567
3,2019,4624
4,2020,5783
5,2021,6990
6,2022,7572
7,2023,7229
8,2024,7493
9,2025,7005


### Important Archive-Coverage Note

The first and last years are partial in the project snapshot:

- 2016 begins in December
- 2026 currently ends in September

The City Traffic Incidents source is also an unofficial archive that can contain collection gaps. In this snapshot, some long gaps occur inside the 2017–2025 calendar window.

Therefore:

- annual values are **recorded archive counts**, not exposure-normalized rates
- 2016 and 2026 should not be compared directly with interior calendar years
- 2017–2025 can still be shown descriptively, but should not be called fully observed years
- source-gap-aware exposure handling is introduced later in Notebook 04


In [9]:
# Plot yearly incident counts.
fig = px.line(
    yearly_counts,
    x="year",
    y="incident_count",
    markers=True,
    title="Recorded Traffic Incidents by Year",
    labels={
        "year": "Year",
        "incident_count": "Recorded incidents",
    },
)

fig.update_layout(
    xaxis=dict(dtick=1)
)

fig.show()

In [10]:
# Keep the main interior calendar-year window for descriptive comparison.
# These are not assumed to have perfect source coverage.
calendar_year_counts = yearly_counts[
    yearly_counts["year"].between(2017, 2025)
].copy()

display(calendar_year_counts)

,year,incident_count
1,2017,5273
2,2018,6567
3,2019,4624
4,2020,5783
5,2021,6990
6,2022,7572
7,2023,7229
8,2024,7493
9,2025,7005


## Monthly Patterns

Monthly counts help identify broad seasonal patterns in recorded
incidents.

In [11]:
# Count incidents by calendar month.
monthly_counts = (
    incidents
    .groupby(["month", "month_name"])
    .size()
    .reset_index(name="incident_count")
    .sort_values("month")
)

display(monthly_counts)

,month,month_name,incident_count
0,1,January,5357
1,2,February,5854
2,3,March,5305
3,4,April,4353
4,5,May,4710
5,6,June,4745
6,7,July,5270
7,8,August,5106
8,9,September,5710
9,10,October,5799


In [12]:
# Plot monthly incident counts.
fig = px.bar(
    monthly_counts,
    x="month_name",
    y="incident_count",
    title="Recorded Traffic Incidents by Month",
    labels={
        "month_name": "Month",
        "incident_count": "Recorded incidents",
    },
)

fig.update_xaxes(
    categoryorder="array",
    categoryarray=monthly_counts["month_name"]
)

fig.show()

## Weekday Patterns

Examine whether incident frequency differs across the seven days of
the week.

In [13]:
# Keep weekdays in natural order.
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_counts = (
    incidents["weekday"]
    .value_counts()
    .reindex(weekday_order)
    .rename_axis("weekday")
    .reset_index(name="incident_count")
)

display(weekday_counts)

,weekday,incident_count
0,Monday,9123
1,Tuesday,10461
2,Wednesday,10840
3,Thursday,10283
4,Friday,10634
5,Saturday,7674
6,Sunday,5783


In [14]:
# Plot weekday incident counts.
fig = px.bar(
    weekday_counts,
    x="weekday",
    y="incident_count",
    title="Recorded Traffic Incidents by Weekday",
    labels={
        "weekday": "Weekday",
        "incident_count": "Recorded incidents",
    },
)

fig.show()

## Hour-of-Day Patterns

Hour-of-day analysis is based on Calgary local time, not UTC.

In [15]:
# Count incidents by hour.
hourly_counts = (
    incidents
    .groupby("hour")
    .size()
    .reset_index(name="incident_count")
)

display(hourly_counts)

,hour,incident_count
0,0,955
1,1,799
2,2,606
3,3,515
4,4,538
5,5,710
6,6,1924
7,7,3578
8,8,3975
9,9,3333


In [16]:
# Plot hourly incident counts.
fig = px.line(
    hourly_counts,
    x="hour",
    y="incident_count",
    markers=True,
    title="Recorded Traffic Incidents by Hour of Day",
    labels={
        "hour": "Hour",
        "incident_count": "Recorded incidents",
    },
)

fig.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig.show()

## Time-Period Patterns

The project uses the following analytical time bins:

- Night: 00:00–05:59
- Morning Rush: 06:00–09:59
- Midday: 10:00–14:59
- Evening Rush: 15:00–18:59
- Evening: 19:00–23:59

These are project-defined categories, not official City of Calgary
rush-hour definitions.

In [17]:
# Count incidents by project-defined time period.
time_period_order = [
    "Night",
    "Morning Rush",
    "Midday",
    "Evening Rush",
    "Evening",
]

time_period_counts = (
    incidents["time_period"]
    .value_counts()
    .reindex(time_period_order)
    .rename_axis("time_period")
    .reset_index(name="incident_count")
)

display(time_period_counts)

,time_period,incident_count
0,Night,4123
1,Morning Rush,12810
2,Midday,16370
3,Evening Rush,21151
4,Evening,10344


In [18]:
# Plot incident counts by time period.
fig = px.bar(
    time_period_counts,
    x="time_period",
    y="incident_count",
    title="Recorded Traffic Incidents by Time Period",
    labels={
        "time_period": "Time period",
        "incident_count": "Recorded incidents",
    },
)

fig.show()

## Weekday vs Weekend

Raw totals are descriptive only because there are five weekdays and
two weekend days in a normal week.

In [19]:
# Compare weekday and weekend totals.
weekend_counts = (
    incidents["weekend"]
    .map({
        False: "Weekday",
        True: "Weekend",
    })
    .value_counts()
    .rename_axis("day_type")
    .reset_index(name="incident_count")
)

display(weekend_counts)

,day_type,incident_count
0,Weekday,51341
1,Weekend,13457


In [20]:
# Plot weekday vs weekend totals.
fig = px.bar(
    weekend_counts,
    x="day_type",
    y="incident_count",
    title="Weekday vs Weekend Recorded Incidents",
    labels={
        "day_type": "Day type",
        "incident_count": "Recorded incidents",
    },
)

fig.show()

## Seasonal Patterns

Seasonal totals provide a broad view before weather data is added later.

In [21]:
# Count incidents by season.
season_order = [
    "Winter",
    "Spring",
    "Summer",
    "Fall",
]

season_counts = (
    incidents["season"]
    .value_counts()
    .reindex(season_order)
    .rename_axis("season")
    .reset_index(name="incident_count")
)

display(season_counts)

,season,incident_count
0,Winter,17805
1,Spring,14368
2,Summer,15121
3,Fall,17504


In [22]:
# Plot seasonal incident counts.
fig = px.bar(
    season_counts,
    x="season",
    y="incident_count",
    title="Recorded Traffic Incidents by Season",
    labels={
        "season": "Season",
        "incident_count": "Recorded incidents",
    },
)

fig.show()

## Incident Descriptions

The source dataset contains detailed text descriptions. Some descriptions
represent essentially the same type of incident using slightly different
wording.

For now, the original descriptions are preserved instead of aggressively
reclassifying them.

In [23]:
# Find the most common source descriptions.
top_descriptions = (
    incidents["description"]
    .value_counts()
    .head(15)
    .rename_axis("description")
    .reset_index(name="incident_count")
)

display(top_descriptions)

,description,incident_count
0,Traffic incident.,17249
1,Two vehicle incident.,8028
2,Traffic incident. Blocking the right lane,3561
3,Single vehicle incident.,2410
4,Two vehicle incident. Blocking the right lane,2266
5,Traffic incident. Blocking the left lane,1603
6,Multi-vehicle incident.,1568
7,Two vehicle incident. Blocking the left lane,1190
8,Multi-vehicle incident. Blocking the right lane,925
9,There is an incident involving a pedestrian- E...,862


In [24]:
# Plot the 15 most common descriptions.
fig = px.bar(
    top_descriptions.sort_values("incident_count"),
    x="incident_count",
    y="description",
    orientation="h",
    title="15 Most Common Incident Descriptions",
    labels={
        "description": "Description",
        "incident_count": "Recorded incidents",
    },
)

fig.show()

## Quadrant Distribution

Quadrant totals show where recorded incidents appear in the source data.
They are not normalized for road length, traffic volume, or population.

In [25]:
# Include UNKNOWN so missing source values remain visible.
quadrant_counts = (
    incidents["quadrant"]
    .value_counts()
    .rename_axis("quadrant")
    .reset_index(name="incident_count")
)

display(quadrant_counts)

,quadrant,incident_count
0,SE,15833
1,UNKNOWN,14058
2,NE,13749
3,NW,11076
4,SW,10082


In [26]:
# Plot incident counts by quadrant.
fig = px.bar(
    quadrant_counts,
    x="quadrant",
    y="incident_count",
    title="Recorded Traffic Incidents by Calgary Quadrant",
    labels={
        "quadrant": "Quadrant",
        "incident_count": "Recorded incidents",
    },
)

fig.show()

## Geographic Distribution

Before applying DBSCAN clustering, verify that incident coordinates form
a plausible Calgary-area distribution and that there are no obvious
coordinate errors.

In [27]:
# Summarize incident coordinate ranges.
coordinate_summary = incidents[
    ["latitude", "longitude"]
].describe(
    percentiles=[
        0.001,
        0.01,
        0.50,
        0.99,
        0.999,
    ]
)

display(coordinate_summary)

,latitude,longitude
count,64798.000000,64798.000000
mean,51.043543,-114.048937
std,0.069035,0.072030
min,50.848554,-114.309770
0.1%,50.861735,-114.257397
1%,50.879401,-114.230226
50%,51.048147,-114.047478
99%,51.175851,-113.918574
99.9%,51.184091,-113.888660
max,51.198505,-113.865579


In [28]:
# Sample points only for visualization.
map_sample = incidents.sample(
    n=min(15000, len(incidents)),
    random_state=42,
)

fig = px.scatter(
    map_sample,
    x="longitude",
    y="latitude",
    hover_data=[
        "incident_location",
        "year",
        "weekday",
        "hour",
    ],
    title="Geographic Distribution of Traffic Incidents",
    labels={
        "longitude": "Longitude",
        "latitude": "Latitude",
    },
    opacity=0.35,
    render_mode="webgl",
)

fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

fig.show()

## Incidents and Traffic Controls

Overlay the cleaned traffic controls on a sample of incident locations
to verify that both datasets occupy the expected Calgary area.

In [29]:
# Plot incidents and traffic controls together.
fig = go.Figure()

fig.add_trace(
    go.Scattergl(
        x=map_sample["longitude"],
        y=map_sample["latitude"],
        mode="markers",
        name="Traffic incidents",
        marker={
            "size": 3,
            "opacity": 0.25,
        },
        text=map_sample["incident_location"],
        hovertemplate="%{text}<extra></extra>",
    )
)

fig.add_trace(
    go.Scattergl(
        x=signals["longitude"],
        y=signals["latitude"],
        mode="markers",
        name="Traffic controls",
        marker={
            "size": 5,
            "opacity": 0.7,
        },
        text=signals["intersection_name"],
        hovertemplate="%{text}<extra></extra>",
    )
)

fig.update_layout(
    title="Traffic Incidents and Traffic Controls",
    xaxis_title="Longitude",
    yaxis_title="Latitude",
)

fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

fig.show()

## Basic Descriptive Findings

Extract simple highest-count categories directly from the data instead
of writing them manually.

In [30]:
# Identify a few descriptive maxima.
busiest_calendar_year = (
    calendar_year_counts
    .loc[
        calendar_year_counts[
            "incident_count"
        ].idxmax()
    ]
)

busiest_month = (
    monthly_counts
    .loc[
        monthly_counts["incident_count"].idxmax()
    ]
)

busiest_weekday = (
    weekday_counts
    .loc[
        weekday_counts["incident_count"].idxmax()
    ]
)

busiest_hour = (
    hourly_counts
    .loc[
        hourly_counts["incident_count"].idxmax()
    ]
)

busiest_time_period = (
    time_period_counts
    .loc[
        time_period_counts[
            "incident_count"
        ].idxmax()
    ]
)

print(
    "Highest recorded count in the 2017–2025 calendar window:",
    int(busiest_calendar_year["year"]),
    "-",
    int(busiest_calendar_year["incident_count"]),
    "incidents",
)

print(
    "Highest-count month:",
    busiest_month["month_name"],
    "-",
    int(busiest_month["incident_count"]),
    "incidents",
)

print(
    "Highest-count weekday:",
    busiest_weekday["weekday"],
    "-",
    int(busiest_weekday["incident_count"]),
    "incidents",
)

print(
    "Highest-count hour:",
    f"{int(busiest_hour['hour']):02d}:00",
    "-",
    int(busiest_hour["incident_count"]),
    "incidents",
)

print(
    "Highest-count time period:",
    busiest_time_period["time_period"],
    "-",
    int(busiest_time_period["incident_count"]),
    "incidents",
)

Highest recorded count in the 2017–2025 calendar window: 2022 - 7572 incidents
Highest-count month: December - 6594 incidents
Highest-count weekday: Wednesday - 10840 incidents
Highest-count hour: 17:00 - 6068 incidents
Highest-count time period: Evening Rush - 21151 incidents


## EDA Summary

This notebook established the basic temporal and geographic structure of
the Calgary traffic incident dataset.

The analysis examined:

- annual incident counts
- monthly patterns
- weekday patterns
- hourly patterns
- project-defined rush-hour periods
- weekday versus weekend totals
- seasonal patterns
- common source incident descriptions
- quadrant distribution
- geographic coordinate coverage
- spatial overlap between incidents and traffic controls

### Interpretation Limitations

These results describe **recorded incident frequency**.

They should not yet be interpreted as direct measures of road danger or
collision probability because the dataset does not currently account for:

- traffic volume
- number of vehicles travelling each road
- road length
- intersection exposure
- weather exposure
- population differences between areas

The next stage will use geographic clustering to identify recurring
incident concentrations rather than relying only on raw visual density.